# Cloudflare R2 Uploader (Colab)

Uploads one or more `.zip` files to your Cloudflare R2 bucket using the S3 API.

- Shows progress, MB/s, and ETA
- Does not store secrets in the notebook

Bucket: `quest-archive`

In [ ]:
!pip -q install boto3

In [ ]:
import os
import time
import math
from getpass import getpass

import boto3
from boto3.s3.transfer import TransferConfig
from google.colab import files

R2_BUCKET_NAME = os.environ.get("R2_BUCKET_NAME", "quest-archive")
R2_ENDPOINT = os.environ.get("R2_ENDPOINT") or input("R2_ENDPOINT (e.g. https://<account_id>.r2.cloudflarestorage.com): ").strip()
R2_ACCESS_KEY_ID = os.environ.get("R2_ACCESS_KEY_ID") or input("R2_ACCESS_KEY_ID: ").strip()
R2_SECRET_ACCESS_KEY = os.environ.get("R2_SECRET_ACCESS_KEY") or getpass("R2_SECRET_ACCESS_KEY: ")

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)

print("Ready. Bucket:", R2_BUCKET_NAME)
print("Endpoint:", R2_ENDPOINT)

In [ ]:
uploaded = files.upload()
if not uploaded:
  raise SystemExit("No file selected")

zip_files = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
if not zip_files:
  raise SystemExit("Please upload at least one .zip")

print("Selected .zip files:")
for f in zip_files:
  print("-", f)

In [ ]:
def _fmt_mb_per_s(bps: float) -> str:
    return f"{bps / (1024 * 1024):.2f} MB/s"

def _fmt_eta(seconds: float) -> str:
    if seconds <= 0 or math.isinf(seconds) or math.isnan(seconds):
        return "--:--"
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m:02d}:{s:02d}"

class ProgressPrinter:
    def __init__(self, total_bytes: int, label: str):
        self.total = int(total_bytes)
        self.label = label
        self.start = time.time()
        self.last_print = 0.0
        self.seen = 0

    def __call__(self, bytes_amount):
        self.seen += int(bytes_amount)
        now = time.time()
        if now - self.last_print < 0.25 and self.seen < self.total:
            return
        self.last_print = now
        elapsed = max(now - self.start, 1e-6)
        speed_bps = self.seen / elapsed
        remaining = max(self.total - self.seen, 0)
        eta = remaining / max(speed_bps, 1e-6)
        pct = (self.seen / self.total * 100.0) if self.total else 0.0
        line = f"{self.label}  {pct:6.2f}%  {_fmt_mb_per_s(speed_bps)}  ETA {_fmt_eta(eta)}"
        print("\r" + line + " " * 10, end="")
        if self.seen >= self.total:
            print()

In [ ]:
config = TransferConfig(
    multipart_threshold=64 * 1024 * 1024,
    multipart_chunksize=64 * 1024 * 1024,
    max_concurrency=8,
    use_threads=True,
)

prefix = input("Optional folder prefix in bucket (blank for root): ").strip()
if prefix:
    prefix = prefix.strip("/") + "/"

for filename in zip_files:
    local_path = filename
    object_key = f"{prefix}{filename}"
    total_size = os.path.getsize(local_path)
    cb = ProgressPrinter(total_size, f"Uploading {filename}")
    s3.upload_file(local_path, R2_BUCKET_NAME, object_key, Config=config, Callback=cb)
    print("Uploaded to:", object_key)